# 02. Generación de pseudo-LiDAR para varios samples

Este notebook corresponde al segundo paso del experimento.

En esta etapa:
- se toma el manifest definido en el paso 1,
- se genera una pseudo-LiDAR para cada sample incluido en dicho manifest,
- y se guarda un resultado independiente por frame.

Justificación:
- para estudiar una aplicación orientada a SLAM no es suficiente con un único frame,
- sino que es necesario disponer de una pseudo-LiDAR por timestamp para poder comparar `t` con `t+1`.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys


In [2]:
ROOT = Path('/home/clara/ml-depth-pro/slam_readiness_nuscenes')
MANIFEST_PATH = ROOT / 'manifests' / 'scene-0061_first5.json'
SCRIPT_PATH = ROOT / 'scripts' / 'generate_pseudolidar_manifest.py'
DEPTH_ANYTHING3_SCRIPT = ROOT / 'scripts' / 'generate_depthanything3_sample.py'
DEPTH_MODEL_COMPARISON_SCRIPT = ROOT / 'scripts' / 'compare_depth_models_common_support.py'
OUTPUT_ROOT = ROOT / 'outputs'

DEPTH_PRO_ROOT = Path(os.environ.get('DEPTH_PRO_ROOT', str(ROOT.parent)))
DEPTH_ANYTHING3_ROOT = Path(os.environ.get('DEPTH_ANYTHING3_ROOT', '/home/clara/depth-anything-3'))
DEPTH_ANYTHING3_SAMPLE_DIR = ROOT / 'outputs' / 'scene-0061' / 'depthanything3_sample_000'

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
manifest


{'version': 'v1.0-mini',
 'dataroot': '/home/clara/datasets/nuscenes',
 'scene_name': 'scene-0061',
 'scene_token': 'cc8c0bf57f984915a77078b10eb33198',
 'description': 'Parked truck, construction, intersection, turn left, following a van',
 'num_requested': 5,
 'num_selected': 5,
 'samples': [{'index': 0,
   'sample_token': 'ca9a282c9e77460f8360f564131a8af5',
   'timestamp_s': 1532402927.647951,
   'prev': '',
   'next': '39586f9d59004284a7114a68825e8eec'},
  {'index': 1,
   'sample_token': '39586f9d59004284a7114a68825e8eec',
   'timestamp_s': 1532402928.147847,
   'prev': 'ca9a282c9e77460f8360f564131a8af5',
   'next': '356d81f38dd9473ba590f39e266f54e5'},
  {'index': 2,
   'sample_token': '356d81f38dd9473ba590f39e266f54e5',
   'timestamp_s': 1532402928.698048,
   'prev': '39586f9d59004284a7114a68825e8eec',
   'next': 'e0845f5322254dafadbbed75aaa07969'},
  {'index': 3,
   'sample_token': 'e0845f5322254dafadbbed75aaa07969',
   'timestamp_s': 1532402929.197353,
   'prev': '356d81f38dd9473

## Función del script

Para cada sample incluido en el manifest, el script realiza las siguientes operaciones:
1. carga las 6 cámaras de `nuScenes`,
2. estima la profundidad de cada imagen mediante `Depth Pro`,
3. reconstruye la nube 3D fusionada (`ring`),
4. la convierte a una representación `pseudo-LiDAR`,
5. y guarda un archivo `.ply` por sample.

En esta etapa todavía **no se evalúa SLAM**. El objetivo es preparar la secuencia de nubes que se utilizará en los análisis posteriores.


In [3]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--manifest', str(MANIFEST_PATH),
    '--output-root', str(OUTPUT_ROOT),
    '--depth-pro-root', str(DEPTH_PRO_ROOT),
    '--max-samples', '5',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)
if result.returncode != 0:
    raise RuntimeError(f'La generacion con Depth Pro ha fallado con codigo {result.returncode}')


/home/clara/ml-depth-pro/depthpro_env/bin/python /home/clara/ml-depth-pro/slam_readiness_nuscenes/scripts/generate_pseudolidar_manifest.py --manifest /home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json --output-root /home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs --depth-pro-root /home/clara/ml-depth-pro --max-samples 5
{
  "sample_token": "ca9a282c9e77460f8360f564131a8af5",
  "index": 0,
  "timestamp_s": 1532402927.647951,
  "ring_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_ring_6cams_ego.ply",
  "pseudo_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply",
  "lidar_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply",
  "ring_num_points": 281010,
  "pseudo_num_points": 23534
}
{
  "sample_token": "39586f9d59004284a7114a6

In [4]:
scene_dir = OUTPUT_ROOT / manifest['scene_name']
summary_path = scene_dir / 'run_summary.json'
if summary_path.exists():
    print(summary_path)
    print(summary_path.read_text())
else:
    print('Todavia no existe run_summary.json. Ejecuta antes la celda del script.')


/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/run_summary.json
{
  "scene_name": "scene-0061",
  "manifest_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json",
  "num_processed": 5,
  "samples": [
    {
      "sample_token": "ca9a282c9e77460f8360f564131a8af5",
      "index": 0,
      "timestamp_s": 1532402927.647951,
      "ring_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_ring_6cams_ego.ply",
      "pseudo_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply",
      "lidar_path": "/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply",
      "ring_num_points": 281010,
      "pseudo_num_points": 23534
    },
    {
      "sample_token": "39586f9d59004284a7114a68825e8eec",
      "index": 1,
      "timestamp_s": 153

## Comparación entre Depth Pro y Depth Anything 3

Una vez generada la pseudo-LiDAR principal con `Depth Pro`, se ejecuta también la inferencia de `Depth Anything 3` sobre el mismo sample de `nuScenes`. El objetivo es que la comparación no dependa de una nube `.ply` externa, sino de una salida generada de forma reproducible desde este notebook.

La comparación se realiza en soporte común, limitado a un rango y altura comparables, para evitar que las zonas extremas dominen la métrica. Después se ejecuta `compare_depth_models_common_support.py`, que carga las nubes `.ply` generadas por ambos modelos, aplica el mismo filtrado espacial y recalcula las métricas frente a `LIDAR_TOP`.


In [5]:
cmd = [
    sys.executable,
    str(DEPTH_ANYTHING3_SCRIPT),
    '--manifest', str(MANIFEST_PATH),
    '--output-root', str(OUTPUT_ROOT),
    '--sample-index', '0',
]
if DEPTH_ANYTHING3_ROOT.exists():
    cmd.extend(['--depth-anything-root', str(DEPTH_ANYTHING3_ROOT)])
else:
    print('No se ha encontrado DEPTH_ANYTHING3_ROOT; se usara el paquete depth_anything_3 instalado en el entorno.')

print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print('returncode:', result.returncode)
if result.returncode != 0:
    raise RuntimeError(f'La inferencia con Depth Anything 3 ha fallado con codigo {result.returncode}')


/home/clara/ml-depth-pro/depthpro_env/bin/python /home/clara/ml-depth-pro/slam_readiness_nuscenes/scripts/generate_depthanything3_sample.py --manifest /home/clara/ml-depth-pro/slam_readiness_nuscenes/manifests/scene-0061_first5.json --output-root /home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs --sample-index 0 --depth-anything-root /home/clara/depth-anything-3
[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
[INFO ] Processed Images Done taking 0.10489749908447266 seconds. Shape:  torch.Size([1, 3, 280, 504])
[INFO ] Model Forward Pass Done. Time: 0.7044341564178467 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0019462108612060547 seconds
CAM_FRONT_LEFT: depth_shape=(540, 960) depth_min=2.62 depth_p95=18.21 infer_time_s=0.82
[INFO ] Processed Images Done taking 0.014742374420166016 seconds. Shape:  torch.Size(

In [6]:
depth_cmp_path = ROOT / 'outputs' / 'scene-0061' / 'depth_model_comparison_common_support.json'

cmd = [
    sys.executable,
    str(DEPTH_MODEL_COMPARISON_SCRIPT),
    '--run-summary', str(ROOT / 'outputs' / 'scene-0061' / 'run_summary.json'),
    '--depth-anything-sample-dir', str(DEPTH_ANYTHING3_SAMPLE_DIR),
    '--sample-index', '0',
    '--output', str(depth_cmp_path),
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'La comparacion de modelos ha fallado con codigo {result.returncode}')

depth_cmp = json.loads(depth_cmp_path.read_text(encoding='utf-8'))
depth_rows = []
for model_name, metrics in depth_cmp['models'].items():
    depth_rows.append({'modelo': model_name, **metrics})

import pandas as pd
display(pd.DataFrame(depth_rows).set_index('modelo'))
print(depth_cmp['interpretation'])


/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/depth_model_comparison_common_support.json
{
  "sample_index": 0,
  "sample_token": "ca9a282c9e77460f8360f564131a8af5",
  "support": {
    "r_min": 2.0,
    "r_max": 35.0,
    "z_min": -2.0,
    "z_max": 2.5,
    "bev_resolution": 0.5
  },
  "inputs": {
    "Depth Pro": "outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_pseudolidar_ego.ply",
    "Depth Anything 3": "outputs/scene-0061/depthanything3_sample_000/pcd_pseudolidar_ego.ply",
    "LIDAR_TOP": "outputs/scene-0061/ca9a282c9e77460f8360f564131a8af5/pcd_lidar_top_ego.ply"
  },
  "lidar_stats": {
    "name": "lidar_top_gt_common_support",
    "num_points": 21992,
    "range_min": 2.000042200088501,
    "range_mean": 8.797609329223633,
    "range_median": 6.411016464233398,
    "range_p95": 22.75317173004151
  },
  "models": {
    "Depth Pro": {
      "pseudo_points": 14275,
      "pseudo_to_gt_mean_nn": 0.7578806281089783,
      "pseudo_to_gt_median_nn": 0.51

,pseudo_points,pseudo_to_gt_mean_nn,pseudo_to_gt_median_nn,pseudo_to_gt_p95_nn,gt_to_pseudo_mean_nn,gt_to_pseudo_median_nn,gt_to_pseudo_p95_nn,bev_iou
modelo,,,,,,,,
Depth Pro,14275,0.757881,0.517612,2.370395,1.220202,0.770706,4.057555,0.235894
Depth Anything 3,14250,0.872608,0.631768,2.330519,1.243242,0.790731,4.474667,0.226992


Depth Pro obtiene una geometría ligeramente más consistente que Depth Anything 3 en soporte común para esta pipeline, por lo que se mantiene como baseline principal.


En esta comparación, `Depth Pro` obtiene menor distancia mediana pseudo-LiDAR -> GT, menor distancia mediana GT -> pseudo-LiDAR y mayor IoU BEV que `Depth Anything 3`. Por este motivo, el resto del análisis de SLAM-readiness se construye sobre la pseudo-LiDAR generada con `Depth Pro`.


## Resultado esperado al final de esta etapa

Dentro de `outputs/scene-0061/<sample_token>/` debería aparecer, para cada sample:
- `pcd_ring_6cams_ego.ply`,
- `pcd_pseudolidar_ego.ply`,
- `summary.json`.

Además, a nivel de escena se generará:
- `run_summary.json`.

Estos archivos constituirán la base para el paso 3, centrado en la comparación temporal entre nubes consecutivas.


## Visualizacion 3D de la generacion y comparacion de pseudo-LiDAR

Ademas de las metricas y los ficheros `.ply`, se incluyen visualizaciones interactivas para inspeccionar la nube densa fusionada, la pseudo-LiDAR resultante y la comparacion entre modelos de profundidad. Esto permite ver el paso desde la reconstruccion 3D generada por las camaras hasta una representacion mas parecida a un LiDAR.

Enlaces directos:

- [Nube densa fusionada](../outputs/scene-0061/pointcloud_phase_visualizations/phase_01_dense_ring_6cams.html)
- [Pseudo-LiDAR Depth Pro frente a LiDAR real](../outputs/scene-0061/pointcloud_phase_visualizations/phase_02_pseudolidar_vs_lidar_gt.html)
- [Depth Pro vs Depth Anything 3 vs LiDAR real](../outputs/scene-0061/pointcloud_phase_visualizations/phase_03a_depthpro_depthanything3_lidar.html)


In [7]:
from IPython.display import IFrame, display, Markdown
VIS_DIR = ROOT / 'outputs' / 'scene-0061' / 'pointcloud_phase_visualizations'
display(Markdown('[Abrir indice de visualizaciones 3D](' + str(VIS_DIR / 'index.html') + ')'))
display(IFrame(src=str(VIS_DIR / 'phase_01_dense_ring_6cams.html'), width='100%', height=720))
display(IFrame(src=str(VIS_DIR / 'phase_02_pseudolidar_vs_lidar_gt.html'), width='100%', height=720))
display(IFrame(src=str(VIS_DIR / 'phase_03a_depthpro_depthanything3_lidar.html'), width='100%', height=720))


[Abrir indice de visualizaciones 3D](/home/clara/ml-depth-pro/slam_readiness_nuscenes/outputs/scene-0061/pointcloud_phase_visualizations/index.html)